# 4. Scenarios

This tutorial changes how emissions develop in the future, in two ways:

1. a trajectory that scales the fossil emissions of replacements (B4) and
   end of life (C1–C4) by year;
2. operational electricity (B6) with a constant eGRID2023 factor, against a
   year-by-year Cambium 2024 factor.

!!! note "Illustrative numbers"
    The floor is `steel.json` from
    [`tests/data/inventories/`](https://github.com/jayarehart/DLCA/tree/main/tests/data/inventories).
    The trajectory and the electricity use are made up for this tutorial and
    written inline below. Only the eGRID2023 and Cambium 2024 factors are real
    data, packaged with DLCA.

In [ ]:
import json
from pathlib import Path

# The example inventories live in the repository's tests/data/inventories/.
# Look for them from the folder this notebook runs in, upwards.
here = Path.cwd().resolve()
DATA = next(
    p / "tests" / "data" / "inventories"
    for p in [here, *here.parents]
    if (p / "tests" / "data" / "inventories").is_dir()
)


def load(name):
    """The example file `name`.json as a plain dict."""
    return json.loads((DATA / f"{name}.json").read_text(encoding="utf-8"))

## A trajectory on B4 and C

`b4_c_trajectory` is a multiplier m(y) on study years (decision record
[0018](../decisions/0018-emission-factor-trajectories.md)). This made-up one
falls in a straight line from 1 in year 0 to 0.5 in year 50 and stays there:
a replacement in year 25 is scaled by 0.75, and end of life in year 61 by
0.5. Trajectories need input `schema_version` 4 or later.

In [ ]:
from dlca import assess_building
from dlca.inventory import BuildingInventory, LinearTrajectory

base = load("steel")
base["schema_version"] = 4
as_entered = BuildingInventory.model_validate(base)
falling = as_entered.model_copy(
    update={"b4_c_trajectory": LinearTrajectory(kind="linear", target=0.5, target_year=50)}
)

print("m(y) in years 0, 25, 50, 61:", falling.b4_c_trajectory.multipliers(62)[[0, 25, 50, 61]])

In [ ]:
import pandas as pd

results = {
    "as entered": assess_building(as_entered, horizon=100),
    "falling trajectory": assess_building(falling, horizon=100),
}
modules = ["A1-A3", "B4", "C1", "C2", "C3", "C4"]
pd.DataFrame(
    {
        name: {m: r.by_module[m].gwi_dynamic[100] for m in modules}
        | {"total": r.total.gwi.dynamic[100]}
        for name, r in results.items()
    }
).round(0)

A1–A3 is the same in both: the trajectory touches only B4 and C1–C4. Those
modules are small here, so the total moves by a few per cent.

## B6: eGRID2023 against Cambium

Now add operational electricity: a made-up 100,000 kWh per year for the
floor (about 100 kWh/m²). Energy lines need `schema_version` 5, and Cambium
needs 6 (decision records
[0019](../decisions/0019-operational-energy-b6.md) and
[0021](../decisions/0021-cambium-b6-factors.md)).

- **eGRID2023**: one factor, the same every year, for the subregion RMPA.
- **Cambium 2024**: NLR's region WestConnect_North, its MidCase scenario,
  with construction (study year 0) in 2025. The factor changes year by
  year: history first, then the scenario's projection.

RMPA and WestConnect_North are different regional groupings; they are
paired here only to show the two kinds of factor.

In [ ]:
from dlca.inventory import EnergyUse

base = load("steel")
base["schema_version"] = 6
floor = BuildingInventory.model_validate(base)

lines = {
    "eGRID2023 RMPA": EnergyUse(
        fuel="electricity", amount=100_000, unit="kWh", egrid_subregion="RMPA"
    ),
    "Cambium MidCase": EnergyUse(
        fuel="electricity",
        amount=100_000,
        unit="kWh",
        cambium_gea="WestConnect_North",
        cambium_scenario="MidCase",
        year_zero=2025,
    ),
}
with_b6 = {name: floor.model_copy(update={"energy": [line]}) for name, line in lines.items()}

[`timeline()`](../api/inventory.md#dlca.inventory.timeline) shows B6 year by
year, in years 1 to 60 of the study period:

In [ ]:
import matplotlib.pyplot as plt

from dlca.inventory import timeline

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
b6_results = {}
for name, building in with_b6.items():
    b6 = timeline(building, horizon=100).by_module()["B6"]["co2"]
    ax1.plot(b6 / 1000, label=name)
    b6_results[name] = assess_building(building, horizon=100)
    ax2.plot(b6_results[name].by_module["B6"].gwi_dynamic / 1000, label=name)
ax1.set(title="B6 CO2 per year", xlabel="study year", ylabel="t CO2")
ax2.set(title="B6 GWI, dynamic", xlabel="study year", ylabel="t CO2e")
ax1.legend()
fig.tight_layout()

In [ ]:
pd.DataFrame(
    {
        name: {
            "B6 CO2, years 1-60 [kg]": timeline(with_b6[name], horizon=100)
            .by_module()["B6"]["co2"]
            .sum(),
            "B6 GWI dynamic, year 100 [kg CO2e]": r.by_module["B6"].gwi_dynamic[100],
            "Total GWI dynamic, year 100 [kg CO2e]": r.total.gwi.dynamic[100],
        }
        for name, r in b6_results.items()
    }
).round(0)

With the constant eGRID factor, B6 outweighs the floor's embodied
emissions over 60 years. With this Cambium scenario the grid factor falls
steeply from the late 2020s, so B6 is much smaller. The difference is the
scenario's assumption about the future grid, not a property of the floor.
Other Cambium scenarios (`cambium_scenario`) give other paths.

Packaged factors cover generation only: no transmission and distribution
losses and no upstream emissions.